In [ ]:

import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd, torch
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.1f}с] {m}", flush=True)
log(f"GPU: {torch.cuda.get_device_name(0)}, ядер CPU: {os.cpu_count()}")

os.makedirs("/kaggle/working/src", exist_ok=True); os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
fus = os.path.dirname(glob.glob("/kaggle/input/**/fusion_boost.npz", recursive=True)[0])
for p in glob.glob(fus + "/fusion_*"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
log(f"слитая модель из {fus}")

# Веса разложены по двум датасетам с разными именами: e5 в fp16-наборе под именем
# ce_e516, spec только в fp32-наборе. На скорость это не влияет — модель всё равно
# грузится в fp16, важен лишь состав.
for prefix, tag in (("ce_spec", "ce_spec"), ("ce_e516", "ce_e5")):
    hits = glob.glob(f"/kaggle/input/**/{prefix}__model.safetensors", recursive=True)
    if not hits: raise SystemExit(f"НЕТ ВЕСОВ: {prefix}")
    root = os.path.dirname(hits[0]); dst = f"/kaggle/working/models/{tag}"
    os.makedirs(dst, exist_ok=True)
    for f in ("model.safetensors", "config.json", "tokenizer.json",
              "tokenizer_config.json", "inference_config.json"):
        if not os.path.exists(f"{dst}/{f}"): os.symlink(f"{root}/{prefix}__{f}", f"{dst}/{f}")
hn = os.path.dirname(glob.glob("/kaggle/input/**/ce_hardneg/model.safetensors", recursive=True)[0])
shutil.copytree(hn, "/kaggle/working/models/ce_hardneg", dirs_exist_ok=True)
# Двухбашенная: в выгрузке ядра обучения каталог называется bi_encoder, решение ждёт ce_bi.
bi = os.path.dirname(glob.glob("/kaggle/input/**/bi_encoder/model.safetensors", recursive=True)[0])
shutil.copytree(bi, "/kaggle/working/models/ce_bi", dirs_exist_ok=True)
assert json.load(open("/kaggle/working/models/ce_bi/inference_config.json")).get("kind") == "biencoder"

os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
info = json.load(open("models/fusion_info.json"))
log(f"состав по модели: {info['encoders']}, столбцов {len(info['columns'])}, "
    f"фолд {info['honest_macro']:.6f}")
for e in info["encoders"]:
    assert os.path.exists(f"models/{e}/model.safetensors"), f"нет весов {e}"

big = os.path.dirname(glob.glob("/kaggle/input/**/llm_pairs_2m.parquet", recursive=True)[0])
pairs = pd.read_parquet(big + "/llm_pairs_2m.parquet").sample(40_000, random_state=1).reset_index(drop=True)
items = pd.read_parquet(big + "/llm_items_2m.parquet")
used = pd.unique(np.concatenate([pairs.id1.to_numpy(), pairs.id2.to_numpy()]))
items = items[items.id.isin(used)].reset_index(drop=True)
matches = pairs[["id1", "id2"]]
matches.to_parquet("/kaggle/working/test_matches.parquet", index=False)
items.to_parquet("/kaggle/working/test_items.parquet", index=False)
N = len(matches)
truth = (pairs["target"].to_numpy() > 0).astype(np.int8) if "target" in pairs else None
del pairs, items, matches; gc.collect()
log(f"прогон на {N:,} парах")

from src.pipeline import predict_pipeline
t = time.perf_counter()
predict_pipeline(items_path="/kaggle/working/test_items.parquet",
                 matches_path="/kaggle/working/test_matches.parquet",
                 output_path="/kaggle/working/submit.csv",
                 method="blend")
total = time.perf_counter() - t
out = pd.read_csv("/kaggle/working/submit.csv")
log(f"\nПРОГОН: {total:.1f}с на {N:,} пар, строк {len(out):,}")
assert len(out) == N, "потеряны строки"
assert int(out.predict.isna().sum()) == 0, "есть пустые предсказания"
log(f"  скор: мин {out.predict.min():.4f} макс {out.predict.max():.4f}, "
    f"уникальных {out.predict.nunique():,}")
if truth is not None:
    from sklearn.metrics import average_precision_score
    log(f"  PR-AUC на этой выборке: {average_precision_score(truth, out.predict):.6f}")
log(f"  пересчёт на Private (255 тыс.): {total*255000/N:.0f}с при лимите 780")
log(f"  пересчёт на Public  (110 тыс.): {total*110000/N:.0f}с при лимите 360")
log("  замер на T4; на H100 быстрее станут только стадии видеокарты")
log("СКВОЗНОЙ ПРОГОН ПРОЙДЕН")
